# GSB 5544 — Topic 5.2: HTML and Web Scraping  
*Fill each `____` blank as you work; the ✅ checks ask for a sentence or two.*

## The next 15 minutes

| | Question | Where it lands in PA 5.2 |
|---|---|---|
| **1. Read** | What is HTML made of? (tags, attributes, text) | parts 3 – 5 |
| **2. Find** | How do I locate the tags I want? (`find`, `find_all`, `attrs`) | parts 2 – 11 |
| **3. Collect** | How do rows of a web table become a data frame? (the loop) | part 12, hockey 3 |
| **4. Scale** | Real pages, `pd.read_html`, and many pages | part 1, part 13, hockey 4 |

An API hands you clean JSON. Most websites do not have one — the data is only *on the page*, wrapped in
HTML meant for a browser. **Scraping** is pulling the data back out of that HTML.

In [ ]:
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup

---
## 1. HTML is a tree of tags

Three things to recognise:

- a **tag** marks a piece of the page: `<td> ... </td>`. Tags sit inside other tags — a tree.
- **attributes** are labels on the opening tag: `<table class="stats">`, `<a href="/page2">`.
- the **text** is what sits between the opening and closing tag.

The tags that matter for tables: `<table>` › `<tr>` (a **row**) › `<th>` (a **header** cell) or `<td>` (a
**data** cell). And for links: `<a href="...">`. A small page, as a string:

In [ ]:
html = """
<html><body>
  <h1>Central Coast cities</h1>
  <table class="notes"><tr><td>Populations are estimates.</td></tr></table>
  <table class="stats" id="cities">
    <tr><th>City</th><th>County</th><th>Population</th></tr>
    <tr><td><a href="/wiki/SLO">San Luis Obispo</a></td><td>SLO</td><td>47,063</td></tr>
    <tr><td><a href="/wiki/Santa_Maria">Santa Maria</a>[a]</td><td>SB</td><td>109,707</td></tr>
    <tr><td><a href="/wiki/Paso_Robles">Paso Robles</a></td><td>SLO</td><td>31,490</td></tr>
  </table>
  <ul class="pagination"><li><a href="/cities?page=1">1</a></li><li><a href="/cities?page=2">2</a></li></ul>
</body></html>
"""

---
## 2. Beautiful Soup: parse, then find

`BeautifulSoup` turns the string into a tree you can search. Two search methods do almost everything:

| | returns | use when |
|---|---|---|
| `soup.find("tag")` | the **first** matching tag (or `None`) | you want one thing |
| `soup.find_all("tag")` | a **list** of every matching tag | you want to count or loop |

Both accept `attrs={...}` to narrow the search by attribute.

In [ ]:
soup = ____(html, "html.parser")
len(soup.____("table"))                      # how many tables on the page?

In [ ]:
table = soup.find("table", attrs={____: ____})     # the one we want, picked out by its attribute
table.attrs

From a tag you can pull out its **text** (`.text`) and its **attributes** (`.attrs["href"]`). A search
can start from *any* tag, not only from `soup` — `table.find_all("tr")` looks only inside that table.

In [ ]:
rows = table.____("tr")
first_city = rows[1]                     # rows[0] is the header row
cells = first_city.find_all(____)
cells[0].text, cells[2].text, cells[0].find("a").attrs[____]

✅ (a) Why `rows[1]` rather than `rows[0]`? What would `rows[0].find_all("td")` return? (b) For Santa
Maria, `cells[0].text` is `'Santa Maria[a]'`. How does `cells[0].find("a").text` avoid the footnote?

**Your answer:** *(write it here — replace this line)*

---
## 3. From rows to a data frame: the loop

Work out the extraction for **one** row, then put it in a loop: empty list → one dictionary per row →
`pd.DataFrame`. Scraped values are always **text**, so clean and convert the numbers.

In [ ]:
records = []
for row in table.find_all("tr")[____]:                       # skip the header row
    cells = row.find_all("td")
    records.append({
        "city": cells[0].find("a").text,
        "county": cells[1].text,
        "population": int(cells[2].text.____(",", "")),   # "47,063" -> 47063
    })

df = pd.____(records)
df

✅ Without the `.replace(",", "")`, what would `int("47,063")` do? And if you skipped `int(...)` entirely, what
would `df["population"].sum()` give you?

**Your answer:** *(write it here — replace this line)*

---
## 4. Real pages

**Get the HTML** with `requests`, exactly as with an API — but the reply is a page, so use `.text`, not
`.json()`. **Find the right tags** with your browser: right-click the thing you want → *Inspect*, and read off
the tag and its attributes.

Two habits: check `status_code` (some sites, including Wikipedia, refuse requests that do not identify
themselves — send a `User-Agent` header), and scrape politely: look at the site's terms, and pause between
requests.

In [ ]:
headers = {"User-Agent": "GSB5544 class exercise"}
response = requests.get("https://www.scrapethissite.com/pages/simple/", headers=headers)
response.____

Inspecting that page shows each country in `<div class="col-md-4 country">`, with the name in
`<h3 class="country-name">` and the capital in `<span class="country-capital">`. Matching on one class
(`"country"`) is enough.

In [ ]:
soup = BeautifulSoup(response.____, "html.parser")
countries = soup.find_all("div", attrs={"class": "country"})

records = []
for country in countries:
    records.append({
        "name": country.find("h3", attrs={"class": "country-name"}).text.____(),
        "capital": country.find("span", attrs={"class": "country-capital"}).text,
        "population": int(country.find("span", attrs={"class": "country-population"}).text),
    })

pd.DataFrame(records).head()

✅ Why `.strip()` on the name but not on the capital? (Look at the page source, or try it without.)

**Your answer:** *(write it here — replace this line)*

**Shortcut for tables.** `pd.read_html` finds `<table>` tags and converts them for you. It returns a **list**
of data frames (one per table), so narrow it with `attrs=` or `match=` and then pick with `[0]`. Pass it HTML
you downloaded yourself, wrapped in `StringIO`:

In [ ]:
from io import StringIO

tables = pd.____(StringIO(html), attrs={"id": "cities"})
len(tables), tables[0]

**Many pages.** When the data is spread over pages, collect the page links (`<a>` tags in the pagination
list), then loop: request → parse → extract → `time.sleep`. It is Topic 5.1's request loop with scraping
inside.

In [ ]:
pagination = BeautifulSoup(html, "html.parser").find("ul", attrs={"class": "pagination"})
[link.attrs["href"] for link in pagination.find_all(____)]

✅ These `href`s start with `/`. What must you add before passing one to `requests.get`?

**Your answer:** *(write it here — replace this line)*

## The four lines to keep

| | |
|---|---|
| **Read** | HTML = tags in a tree, with attributes and text; tables are `table › tr › th/td`; links are `a` with `href` |
| **Find** | `BeautifulSoup(html, "html.parser")`; `find` = first, `find_all` = list; narrow with `attrs={...}`; then `.text`, `.attrs[...]` |
| **Collect** | solve one row → loop → list of dictionaries → `pd.DataFrame`; skip header rows; clean and convert numbers |
| **Scale** | `requests.get(url, headers=...)` + check `status_code`; `pd.read_html` for whole tables; loop over page links with `time.sleep` |

PA 5.2 is on the course site: [https://gato365.github.io/gsb5544_instructor_learn_prep/](https://gato365.github.io/gsb5544_instructor_learn_prep/).